# Laboratorio 1 — Ejercicio 2

Encontrar todos los ceros de

$$f(x)=e^{-x^2}+x^4-4x^2+1$$

con una precisión de 6 cifras decimales. Para cada raíz se debe indicar el algoritmo, el punto inicial y el número de iteraciones.

## Resumen del ejercicio

Este ejercicio requiere encontrar todos los ceros reales de una función no polinómica y documentar, para cada uno, el método, el punto inicial y el costo iterativo. La estrategia consiste en estudiar primero la simetría y el número posible de raíces, después se aplica Newton–Raphson desde cuatro puntos iniciales seleccionados de acuerdo con cambios de signo.

Se demuestra que existen exactamente cuatro ceros. Con seis decimales son $-1.930046$, $-0.678806$, $0.678806$ y $1.930046$. Los cuatro se obtuvieron mediante Newton–Raphson en 4 iteraciones, usando respectivamente $x_0=-2$, $-0.5$, $0.5$ y $2$. Los residuos finales son nulos a precisión de máquina o del orden de $10^{-15}$.

## Objetivo y referencias

Se utiliza Newton–Raphson, siguiendo la relación de recurrencia presentada en clase:

$$x_{k+1}=x_k-\frac{f(x_k)}{f'(x_k)}.$$



## 1. Función y derivada

La derivada necesaria para Newton es

$$f'(x)=-2xe^{-x^2}+4x^3-8x.$$

La función es par, pues $f(-x)=f(x)$. Por ello, sus raíces reales aparecen en pares simétricos.

In [1]:
import math

def f(x):
    return math.exp(-x**2) + x**4 - 4*x**2 + 1

def df(x):
    return -2*x*math.exp(-x**2) + 4*x**3 - 8*x

# Función auxiliar obtenida mediante y = x².
def g(y):
    return math.exp(-y) + y**2 - 4*y + 1

def d2g(y):
    return math.exp(-y) + 2

### Explicación de esta parte:

La función `f(x)` representa el problema original y `df(x)` implementa su derivada analítica. Newton–Raphson necesita ambas evaluaciones en cada paso: $f(x_k)$ mide qué tan lejos está la aproximación de satisfacer la ecuación y $f'(x_k)$ proporciona la pendiente de la recta tangente usada para corregirla.

Las funciones `g(y)` y `d2g(y)` no forman parte de las iteraciones de Newton. Se introducen para el argumento de exhaustividad después de hacer el cambio $y=x^2$. En particular, `d2g(y)` permite comprobar que $g''(y)>0$ y, por lo tanto, que $g$ es estrictamente convexa.

## 2. Aislamiento de los ceros

Los valores siguientes muestran cambios de signo en $[-2,-1]$, $[-1,0]$, $[0,1]$ y $[1,2]$. Estos intervalos permiten escoger puntos iniciales cercanos sin comenzar en $x_0=0$, donde la derivada es nula.

In [2]:
print(f"f(-2) = {f(-2): .12f}")
print(f"f(-1) = {f(-1): .12f}")
print(f"f(0)  = {f(0): .12f}")
print(f"f(1)  = {f(1): .12f}")
print(f"f(2)  = {f(2): .12f}")

f(-2) =  1.018315638889
f(-1) = -1.632120558829
f(0)  =  2.000000000000
f(1)  = -1.632120558829
f(2)  =  1.018315638889


### Interpretación de las evaluaciones:

Los signos alternan como positivo, negativo, positivo, negativo y positivo al recorrer $-2,-1,0,1,2$. Debido a la continuidad de $f$, esto garantiza al menos una raíz en cada uno de los cuatro intervalos contiguos. La igualdad $f(-x)=f(x)$ se observa también numéricamente: los valores en $-2$ y $2$ coinciden, al igual que los valores en $-1$ y $1$.

Aislar raíces y aproximarlas son tareas diferentes. Estos cambios de signo indican dónde buscarlas, pero todavía no determinan su valor con seis decimales. Además, $x_0=0$ no se usa como inicio de Newton porque $f'(0)=0$ causaría una división por cero en la fórmula de actualización.

### ¿Por qué existen exactamente cuatro ceros?

Con el cambio $y=x^2$, donde $y\geq0$, la ecuación se transforma en

$$g(y)=e^{-y}+y^2-4y+1=0.$$

Se tiene $g(0)=2>0$, $g(1)=e^{-1}-2<0$ y $g(4)=e^{-4}+1>0$. Por continuidad, existe una raíz de $g$ en $(0,1)$ y otra en $(1,4)$. Además,

$$g''(y)=e^{-y}+2>0,$$

por lo que $g$ es estrictamente convexa y no puede tener más de dos ceros. Como ambos valores de $y$ son positivos, cada uno produce dos soluciones $x=\pm\sqrt{y}$. Por tanto, $f$ tiene exactamente cuatro raíces reales.

## 3. Implementación de Newton–Raphson

Para garantizar el redondeo a seis decimales se usa $\varepsilon=0.5\times10^{-6}$. El criterio de convergencia es $|x_{k+1}-x_k|<\varepsilon$ y se fija un máximo de 100 iteraciones. También se detiene el algoritmo si la derivada es demasiado pequeña.

In [3]:
def newton_raphson(funcion, derivada, x0, tolerancia=0.5e-6, max_iteraciones=100):
    x = x0
    historial = []

    for iteracion in range(1, max_iteraciones + 1):
        dfx = derivada(x)
        if abs(dfx) < 1e-15:
            raise ZeroDivisionError("Derivada nula o demasiado pequeña.")

        x_nuevo = x - funcion(x) / dfx
        error = abs(x_nuevo - x)
        residuo = abs(funcion(x_nuevo))
        historial.append((iteracion, x, x_nuevo, error, residuo))

        if error < tolerancia:
            return x_nuevo, historial

        x = x_nuevo

    raise RuntimeError("No se alcanzó la convergencia.")

### Explicación del algoritmo implementado:

La función `newton_raphson` comienza con $x=x_0$. En cada iteración evalúa la derivada, calcula $x_{k+1}=x_k-f(x_k)/f'(x_k)$ y registra el error absoluto $|x_{k+1}-x_k|$ junto con el residuo $|f(x_{k+1})|$. Geométricamente, $x_{k+1}$ es la intersección con el eje horizontal de la recta tangente trazada en $(x_k,f(x_k))$.

El algoritmo termina cuando el cambio entre aproximaciones es menor que la tolerancia. La condición `abs(dfx) < 1e-15` evita dividir por una pendiente prácticamente nula, mientras que `max_iteraciones=100` protege contra divergencia o ciclos. El historial se conserva para poder auditar cómo se alcanzó cada raíz, no solamente mostrar el valor final.

## 4. Resultados

Se eligen los puntos iniciales $-2$, $-0.5$, $0.5$ y $2$, uno para cada raíz aislada.

In [4]:
tolerancia = 0.5e-6
puntos_iniciales = [-2.0, -0.5, 0.5, 2.0]
resultados = []

for x0 in puntos_iniciales:
    raiz, historial = newton_raphson(f, df, x0, tolerancia)
    resultados.append((raiz, x0, historial))

resultados.sort(key=lambda resultado: resultado[0])

print(f"{'raíz':>11} {'x0':>7} {'iteraciones':>12} {'residuo':>13}")
for raiz, x0, historial in resultados:
    print(f"{raiz:11.6f} {x0:7.1f} {len(historial):12d} {abs(f(raiz)):13.3e}")

       raíz      x0  iteraciones       residuo
  -1.930046    -2.0            4     1.776e-15
  -0.678806    -0.5            4     0.000e+00
   0.678806     0.5            4     0.000e+00
   1.930046     2.0            4     1.776e-15


### Interpretación de la tabla resumen:

Cada fila relaciona explícitamente una raíz con el punto desde el cual comenzó Newton. Los puntos $-2$ y $2$ conducen a las raíces exteriores, mientras que $-0.5$ y $0.5$ conducen a las interiores. Esta selección respeta los intervalos aislados previamente y evita la derivada nula del origen.

Los cuatro procesos terminan en 4 iteraciones. Los residuos de las raíces interiores aparecen como `0.000e+00` porque, con aritmética de punto flotante, la sustitución produce exactamente cero a la precisión disponible. En las raíces exteriores el residuo es $1.776\times10^{-15}$, también despreciable frente a la tolerancia $0.5\times10^{-6}$. La simetría de los resultados es una comprobación independiente de consistencia.

### Historial de convergencia

Las tablas registran cada actualización, el error absoluto por paso y el residuo.

In [5]:
for raiz, x0, historial in resultados:
    print()
    print(f"x0 = {x0:.1f}  ->  raíz = {raiz:.6f}")
    print(f"{'k':>2} {'x_k':>14} {'x_k+1':>14} {'error':>12} {'residuo':>12}")
    for k, x, x_nuevo, error, residuo in historial:
        print(f"{k:2d} {x:14.9f} {x_nuevo:14.9f} {error:12.3e} {residuo:12.3e}")


x0 = -2.0  ->  raíz = -1.930046
 k            x_k          x_k+1        error      residuo
 1   -2.000000000   -1.936062509    6.394e-02    8.024e-02
 2   -1.936062509   -1.930095773    5.967e-03    6.621e-04
 3   -1.930095773   -1.930045714    5.006e-05    4.638e-08
 4   -1.930045714   -1.930045711    3.507e-09    1.776e-15

x0 = -0.5  ->  raíz = -0.678806
 k            x_k          x_k+1        error      residuo
 1   -0.500000000   -0.696620695    1.966e-01    9.010e-02
 2   -0.696620695   -0.678878602    1.774e-02    3.670e-04
 3   -0.678878602   -0.678805723    7.288e-05    6.821e-09
 4   -0.678805723   -0.678805722    1.355e-09    0.000e+00

x0 = 0.5  ->  raíz = 0.678806
 k            x_k          x_k+1        error      residuo
 1    0.500000000    0.696620695    1.966e-01    9.010e-02
 2    0.696620695    0.678878602    1.774e-02    3.670e-04
 3    0.678878602    0.678805723    7.288e-05    6.821e-09
 4    0.678805723    0.678805722    1.355e-09    0.000e+00

x0 = 2.0  ->  raí

### Cómo leer las tablas de iteración

La columna $x_k$ contiene la aproximación antes de aplicar Newton y $x_{k+1}$ la aproximación corregida. `error` mide el tamaño del paso y `residuo` mide qué tan cerca está el nuevo valor de cumplir $f(x)=0$. Ambos indicadores disminuyen rápidamente.

Por ejemplo, para $x_0=2$ el error pasa de $6.394\times10^{-2}$ a $5.967\times10^{-3}$, luego a $5.006\times10^{-5}$ y finalmente a $3.507\times10^{-9}$. Esta reducción acelerada es el comportamiento esperado de la convergencia cuadrática de Newton cerca de una raíz simple. Las tablas negativas son el reflejo de las positivas porque la función es par y su derivada es impar.

Aunque en la tercera iteración los residuos ya son muy pequeños, el criterio definido se basa en el cambio entre aproximaciones. Por eso se realiza una cuarta iteración, que confirma la estabilidad de los seis decimales reportados.

## Resultado final:

Con seis cifras decimales, todos los ceros son

$$\boxed{x_1=-1.930046},\quad\boxed{x_2=-0.678806},\quad\boxed{x_3=0.678806},\quad\boxed{x_4=1.930046}.$$

Para las cuatro raíces se utilizó **Newton–Raphson**. Los puntos iniciales fueron, respectivamente, $-2$, $-0.5$, $0.5$ y $2$, y en los cuatro casos se alcanzó el criterio de convergencia en **4 iteraciones**.

### Conclusión:

La solución combina una demostración global y un cálculo local. La sustitución $y=x^2$, los cambios de signo y la convexidad demuestran que no existen más de cuatro raíces; Newton–Raphson determina sus valores con la precisión solicitada. Los residuos finales y la simetría numérica confirman la consistencia del resultado. Así, el informe no depende únicamente de que el algoritmo haya convergido: también explica por qué las cuatro soluciones obtenidas constituyen el conjunto completo de ceros reales.